# Embedding responses across organoid size

**Role:** Analysis.

Inspect final node representations and curvature readouts while varying supplied organoid size on fixed graphs. A single PCA fitted at an explicitly chosen reference N provides comparable coordinates for all states. Optional single-cell edits expose displacement vectors without assigning a universal biological meaning to hidden axes. GIN and FiLM share this workflow; size-independent GNN embeddings may remain fixed even when the head prediction changes. This replaces the historical KI67 N=300-specific PCA notebook.

**What this answers:** For one saved checkpoint and one validation organoid, how does changing only the supplied N move the same cells in embedding space and change their curvature predictions? The shared PCA is fitted on that organoid at `REFERENCE_N` (its observed size by default). The main panels show PCA trajectories, predicted curvature versus N, and full-dimensional embedding displacement. `RECIPIENT_NODES=None` displays the first eight cells. `SOURCE_NODE=None` means no ablation is performed; selecting a source enables additional edit-displacement and edited-minus-intact prediction panels.

This is an exploratory single-organoid probe, not population clustering or an estimate averaged over validation organoids. Use `clustering.ipynb` for cluster composition and anatomy, and the ablation notebooks for sampled population effects. The final optional section only reads historical LGR5/FiLM hidden-response tables; its N=361 reference belongs to that old study and does not control the main PCA analysis.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Analysis settings

Choose the input run or cached report and all analysis options here before model loading, inference or plotting. Derived paths and model-dependent defaults are resolved in the following cells; each checkpoint restores its saved data and transformations.

In [ ]:
import numpy as np
import torch

# Sampling and plot selection
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
GRAPH_INDEX = 0  # Validation organoid index used for the embedding probe.
SWEEP_COUNTS = [80,150,300,550,700]  # Supplied cell counts to evaluate; None uses the saved grid.
REFERENCE_N = None  # Reference count for PCA; None uses this graph’s observed N.
SOURCE_NODE = None  # Local source-cell index to edit; None leaves fates intact.
REPLACEMENT_MARKER = None  # Replacement fate; None uses zero/unassigned.
RECIPIENT_NODES = None  # Optional explicit selection; None uses the notebook's documented default.

# Input runs, data and destinations
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
LGR5_RUN_NAME = '20260910_164825_all_depth2_four_models_baseline_with_size'  # Name of the new run or the explicitly selected saved run.

# Model selection
MODEL_KEY = None  # Optional explicit selection; None uses the notebook's documented default.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
LGR5_DEVICE = 'cuda'  # Inference/training device; CUDA must be available for cuda.
LGR5_BATCH_SIZE = 256  # Number of graphs or subgraphs per batch.

# Execution and display
SHOW_SAVED_HIDDEN_RESPONSES = False  # Display historical FiLM hidden-response summaries.

# Statistical support
LGR5_MIN_ORGANOIDS = 10  # Minimum distinct organoids required for reported effects.
LGR5_BOOTSTRAP_SAMPLES = 1000  # Number of organoid bootstrap resamples.


## Film figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

FILM_ROUTE_LABELS = {
    'full': 'FiLM + head vary',  # Figure label for full.
    'film_only': 'FiLM varies; head fixed',  # Figure label for film_only.
    'head_only': 'Head varies; FiLM fixed',  # Figure label for head_only.
    'layer1_only': 'Only FiLM layer 1 varies',  # Figure label for layer1_only.
    'layer2_only': 'Only FiLM layer 2 varies',  # Figure label for layer2_only.
}  # Display labels for the size-conditioning routes.
FILM_COLORS = dict(zip(FILM_ROUTE_LABELS, ['#222222','#d45a31','#2878b5','#29945b','#9467bd']))  # Display colors for the size-conditioning routes.

def film_axis_style(ax, counts):
    ax.set_xscale('log')
    ax.set_xticks([counts[0],counts[len(counts)//2],counts[-1]],
                  labels=[str(counts[0]),str(counts[len(counts)//2]),str(counts[-1])])
    ax.minorticks_off()
    ax.tick_params(labelsize=8)
    ax.spines[['top','right']].set_visible(False)

def film_hidden_curves(table, markers, counts, *, statistic='cosine', minimum=10):
    routes=['full','head_only','layer1_only','layer2_only']
    fig,axes=plt.subplots(3,len(markers),squeeze=False,figsize=(3.1*len(markers),8.8),sharex=True)
    for row,(layer,hop) in enumerate([('h1',1),('h2',1),('h2',2)]):
        for col,marker in enumerate(markers):
            ax=axes[row,col];ax.axhline(1,color='0.75',lw=.7)
            for route in routes:
                group=table[(table.metric==f'{layer}_{statistic}')&(table.hop==hop)&(table.source_marker_name==marker)&(table.route==route)].sort_values('n')
                valid=group.n_organoids>=minimum
                ax.plot(group.n,group['mean'].where(valid),color=FILM_COLORS[route],lw=1.5)
                ax.fill_between(group.n,group.ci_low.where(valid),group.ci_high.where(valid),color=FILM_COLORS[route],alpha=.09)
            film_axis_style(ax,counts)
            if statistic=='cosine':ax.set_ylim(-.05,1.05)
            if row==0:ax.set_title(marker,fontsize=10)
            if col==0:ax.set_ylabel(f'Layer {layer[-1]}, hop {hop}\n'+('Cosine to N=361 response' if statistic=='cosine' else 'Response norm / norm at N=361'))
            if row==2:ax.set_xlabel('Supplied N')
    fig.legend(handles=[Line2D([],[],color=FILM_COLORS[r],label=FILM_ROUTE_LABELS[r]) for r in routes],loc='upper center',bbox_to_anchor=(.5,.96),ncol=4,fontsize=9)
    fig.suptitle('Marker-induced hidden-state changes at LGR5 centers',y=.998)
    fig.tight_layout(rect=(0,0,1,.92))
    return fig

## Select one checkpoint and reference population

Choose one model and organoid so all comparisons remain paired. The reference N is a visible analysis choice, not a hardcoded biological landmark. The model must have a saved log-cell-count input.

In [ ]:
import json, copy
import pandas as pd
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
if MODEL_KEY is None:
    MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'embedding_responses'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Example selection

# Size sweeps

# Source-cell intervention
assert 'log_num_cells' in selection['global_features']
base_graph=g_val[GRAPH_INDEX]
if REFERENCE_N is None:
    REFERENCE_N = len(base_graph.x)  # Reference count for PCA; None uses this graph’s observed N.
print('Observed N:',len(base_graph.x),'source:',SOURCE_NODE)

## Infer paired hidden states and predictions

Change only the supplied standardized log N. Original topology, other globals and the baseline offset stay fixed. Edited and intact graphs use the same N; zeroing is explicitly an unassigned reference, while named replacements set one fate bit.

In [ ]:
from src.analysis.embeddings.clustering import extract_node_embeddings
from sklearn.decomposition import PCA
size_col=selection['global_features'].index('log_num_cells')
pre_col=selection['all_global_features'].index('log_num_cells')
center=selection['global_center'][pre_col];scale=selection['global_scale'][pre_col]
states={};predictions={}
for N in sorted(set([REFERENCE_N,*SWEEP_COUNTS])):
    for edit in [False,True] if SOURCE_NODE is not None else [False]:
        g=base_graph.clone()
        g.global_feat[:,size_col]=(np.log(N)-center)/scale
        if edit:
            g.x[SOURCE_NODE,:len(marker_names)]=0
            if REPLACEMENT_MARKER is not None:
                g.x[SOURCE_NODE,marker_names.index(REPLACEMENT_MARKER)]=1
        extracted=extract_node_embeddings([g],model,device=DEVICE)
        states[N,edit]=extracted.embeddings_local
        predictions[N,edit]=np.asarray(selection['transform'].inverse(extracted.y_pred)).reshape(-1)+selection['baseline_offsets'][g.organoid_str]
pca=PCA(n_components=min(3,states[REFERENCE_N,False].shape[1],len(base_graph.x)))
pca.fit(states[REFERENCE_N,False])
print('Reference PCA variance captured:',pca.explained_variance_ratio_.sum())

## Movement and readout

Follow the same selected cells in a shared PCA frame, and show curvature predictions alongside full-dimensional displacement norms. Large movements outside the displayed PCs remain visible in the norm panel. These trajectories are model-input sweeps, not tracked biological growth.

In [ ]:
if RECIPIENT_NODES is None:
    RECIPIENT_NODES = list(range(min(8,len(base_graph.x))))  # Local cell indices whose embedding trajectories are shown.
fig,axes=plt.subplots(1,3,figsize=(16,4))
for node in RECIPIENT_NODES:
    trajectory=np.array([pca.transform(states[N,False][node:node+1])[0] for N in SWEEP_COUNTS])
    axes[0].plot(trajectory[:,0],trajectory[:,1] if trajectory.shape[1]>1 else np.zeros(len(trajectory)),'o-',label=str(node))
    axes[1].plot(SWEEP_COUNTS,[predictions[N,False][node] for N in SWEEP_COUNTS],'o-')
    axes[2].plot(SWEEP_COUNTS,[np.linalg.norm(states[N,False][node]-states[REFERENCE_N,False][node]) for N in SWEEP_COUNTS],'o-')
axes[0].set(xlabel='PC1',ylabel='PC2');axes[0].legend(title='Cell')
axes[1].set(xscale='log',xlabel='Supplied N',ylabel='Predicted curvature')
axes[2].set(xscale='log',xlabel='Supplied N',ylabel='Full hidden displacement from reference')
plt.tight_layout();plt.show()
if SOURCE_NODE is not None:
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for N in SWEEP_COUNTS:
        a=pca.transform(states[N,False][RECIPIENT_NODES]);b=pca.transform(states[N,True][RECIPIENT_NODES])
        if a.shape[1]>1:
            axes[0].quiver(a[:,0],a[:,1],b[:,0]-a[:,0],b[:,1]-a[:,1],angles='xy',scale_units='xy',scale=1,label=str(N))
    for node in RECIPIENT_NODES:
        axes[1].plot(SWEEP_COUNTS,[predictions[N,True][node]-predictions[N,False][node] for N in SWEEP_COUNTS],'o-',label=str(node))
    axes[0].set(xlabel='PC1',ylabel='PC2',title='Paired edit displacements');axes[0].legend(title='N')
    axes[1].set(xscale='log',xlabel='N',ylabel='Edited − intact curvature');plt.show()
np.savez_compressed(OUTPUT_DIR/'states.npz',**{f'N{N}_edit{int(edit)}':h for (N,edit),h in states.items()})

## Optional: saved FiLM hidden-response summaries

Read the completed LGR5-recipient diagnostic to inspect hidden response direction and norm across sizes. These plots refer to that recorded cohort; they are distinct from the single-checkpoint exploratory PCA above.

In [ ]:
if SHOW_SAVED_HIDDEN_RESPONSES:
    from IPython.display import Markdown

    from src.analysis.conditioning.film import run_diagnostics
    from src.analysis.conditioning.response_statistics import summarize_diagnostics

    # Study-specific functions are defined in this notebook.

    # Paths and outputs
    LGR5_RUN_DIR = PROJECT_ROOT / 'results_experiments/size_conditioned_ablation' / LGR5_RUN_NAME  # Training run to create or saved run to analyze.
    LGR5_OUTPUT_DIR = LGR5_RUN_DIR / 'lgr5_film_diagnostics'  # Directory for this analysis’s tables and figures.

    # Execution switches

    # Inference resources

    # Statistical summaries
    LGR5_CONFIG = json.loads((LGR5_OUTPUT_DIR / 'settings.json').read_text())  # Settings for the observed-cell or intervention study.
    LGR5_COUNTS = LGR5_CONFIG['counts']  # Supplied count grid recorded in the selected diagnostic.
    LGR5_MARKERS = LGR5_CONFIG['marker_names']  # Marker order recorded by the selected training run.
    lgr5_tables = {name: pd.read_csv(LGR5_OUTPUT_DIR / f'{name}.csv') for name in [
        'sampling_support', 'reproduction_checks', 'center_coexpression_support',
        'effect_summary', 'effect_seed_summary', 'effect_endpoint_summary', 'effect_endpoint_seeds',
        'interaction_summary', 'interaction_seed_summary', 'interaction_endpoint_summary', 'interaction_endpoint_seeds',
        'common_scaling', 'common_scaling_seeds', 'common_scaling_support', 'route_interaction_summary',
    ]}
    LGR5_FIGURE_DIR = LGR5_OUTPUT_DIR / 'figures'  # Destination for the generated figures.
    LGR5_FIGURE_DIR.mkdir(exist_ok=True)
    def lgr5_show_save(fig, name):
        for extension in ['png', 'pdf']:
            fig.savefig(LGR5_FIGURE_DIR / f'{name}.{extension}', dpi=150, bbox_inches='tight')
        display(fig)
        plt.close(fig)
    print('Source run:', LGR5_RUN_NAME)
    display(lgr5_tables['sampling_support'], lgr5_tables['center_coexpression_support'])
    print('Completed checkpoints:', len(lgr5_tables['reproduction_checks']))
    print('Maximum difference from original saved ΔK:', lgr5_tables['reproduction_checks'].max_saved_raw_difference.max())


#### 3. Direction and magnitude of hidden marker responses

For each individual ablation, compute the change in the center representation after each
GNN layer, including ReLU and residual connections, but **before appending global features
to the head**. Compare its direction (cosine similarity) and norm with the same case and
checkpoint at N=361, then average within organoids. Cosine 1 means the same direction;
lower values mean a different combination of hidden features responds to that marker.

A zero response has no defined direction, and is excluded from cosine/ratio summaries.
Hop-2 perturbations cannot reach the center in layer 1, so that combination is omitted.
Hidden coordinates and norms are model-specific: no channel alignment across seeds is
assumed. These measurements support a representation-level change, not a biological
interpretation of individual FiLM channels. Changes in direction can arise from
size-dependent channel scaling as well as nonlinear feature computation.

In [ ]:
if SHOW_SAVED_HIDDEN_RESPONSES:
    for lgr5_statistic in ['cosine','ratio']:
        lgr5_show_save(film_hidden_curves(lgr5_tables['effect_summary'], LGR5_MARKERS, LGR5_COUNTS,
            statistic=lgr5_statistic, minimum=LGR5_MIN_ORGANOIDS), f'hidden_response_{lgr5_statistic}')
